# Regulus — ingest real standards and look up applicable provisions

This notebook demonstrates the **Phase-1 baseline**: ingest real regulatory texts
(EU AI Act from EUR-Lex, NIST AI RMF from NIST), then map a free-text issue to the
**applicable provisions**, each with a source citation.

Regulus is built on the [Geometric Knowledge Network (GKN)](https://github.com/minw0607/geometric_knowledge_network)
retrieval substrate. The default retriever is TF-IDF (no API keys); set
`REGULUS_RETRIEVER=embedding` in `.env` for embedding-quality retrieval.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent / 'src'))

from regulus.config import RegulusConfig
from regulus.standards_loader import StandardsLoader
from regulus.lookup import RegulusLookup

config = RegulusConfig()
print('cache dir:', config.cache_dir)
print('retriever:', config.retriever, '| top_k:', config.top_k)

## 2. Ingest real frameworks

Downloads and caches the source documents on first run (EUR-Lex HTML, NIST PDF),
then parses them into citable `Provision` records. Later runs load from cache.

In [ ]:
provisions = StandardsLoader(config).load(framework_ids=['eu_ai_act', 'nist_ai_rmf'])

from collections import Counter
by_fw = Counter(p.framework_name for p in provisions)
print(dict(by_fw))

# a couple of sample provisions
for p in provisions[:2]:
    print('\n', p.citation())
    print('  source:', p.source_url)
    print('  text:', p.text[:200], '...')

## 3. Build the lookup index

In [ ]:
lookup = RegulusLookup(provisions, config)
print(f'Indexed {len(provisions)} provisions as {len(lookup.chunks)} chunks.')

## 4. Look up applicable provisions for an issue

Try your own governance issue or observation below.

In [ ]:
issues = [
    'Our credit model was deployed without testing for demographic bias across protected groups.',
    'We run real-time facial recognition in public spaces to assist law enforcement.',
    'There is no documentation of the data used to train our high-risk hiring model.',
]

for issue in issues:
    print('=' * 100)
    print('ISSUE:', issue, '\n')
    for i, r in enumerate(lookup.search(issue, top_k=4), 1):
        print(f'{i}. [{r.score:.3f}] {r.provision.citation()}')
        print(f'     source: {r.provision.source_url}')
        print(f'     {r.snippet.strip()[:200]}...')
    print()

## 5. Next steps

This is the retrieval baseline. The value of Regulus grows with the **graph layer**:

- build the regulatory knowledge graph (frameworks · provisions · risks) and **cited crosswalks** across frameworks;
- use GKN's multi-hop retriever to expand an issue → applicable provision → cross-referenced provisions in other frameworks;
- return the **evidence path** (via GKN's path explainer) so every answer is traceable;
- add an LLM interpretation layer that produces a structured answer (risks · standards · cross-refs · guidance · citations).

For embedding-quality retrieval, set `REGULUS_RETRIEVER=embedding` in `.env`.